# Text Representation

Text representation in NLP means converting text into a numerical form that a machine learning model can understand and process.

## What is Embedding?

Embeddings in NLP is a technique where individual words are represented as real-valued vectors and captures inter-word semantics.


In this notebook, We going to intreduce 2 techniques for embedding. These techniques will be used for a machine learning models such as SVM, Random forest, ... ect.

<img src="https://drive.google.com/uc?export=view&id=1jd0u_sGppDKqBYbhtJfGxp14lnUWUTTw" width="900">

# 1- TF-IDF

<img src="https://drive.google.com/uc?export=view&id=1saSt0mMgOQ2ybbTms1lu_ruhUcBWkKzL" width="500">

TF-IDF stands for term frequency-inverse document frequency. It is a measure that discounts common words. Used in the fields of information retrieval (IR) and machine learning, that can quantify the importance of words in a document amongst a collection of documents (also known as a corpus).

### Components of TF-IDF

1. **TF (Term Frequency):**  
   Measures how frequently a term appears in a document.


$$ \text{TF}(t, d) = \frac{\text{Number of times term } t \text{ appears in document } d}{\text{Total number of terms in document } d} $$


2. **IDF (Inverse Document Frequency):**  
   Measures how important a term is across all documents. Words that appear in many documents get lower scores.

$$    \text{IDF}(t) = \log \left(\frac{\text{Number of all documents N}}{\text{Number of documents containing the term } t}\right) $$

<img src="https://drive.google.com/uc?export=view&id=1JqPILC8TTh3yDQZCSuPNXwm6ER5YeJFh" width="900">

In [1]:
!pip install gensim
import pandas as pd
import numpy as np
import re
import nltk
import gensim

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("clmentbisaillon/fake-and-real-news-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'fake-and-real-news-dataset' dataset.
Path to dataset files: /kaggle/input/fake-and-real-news-dataset


In [3]:
df = pd.read_csv('/kaggle/input/fake-and-real-news-dataset/True.csv', engine='python', on_bad_lines='skip') #fake-and-real-news-dataset-true.csv
df.head()

,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


In [4]:
doc_1 = "Data is the oil of the digital economy"
doc_2 = "Data is a new oil"

data = [doc_1, doc_2]


In [5]:
tfidf = TfidfVectorizer()
result = tfidf.fit_transform(data) # returns sparce matrix

In [6]:
tfidf_example_df = pd.DataFrame(result.toarray(), columns=tfidf.get_feature_names_out())
tfidf_example_df

,data,digital,economy,is,new,of,oil,the
0,0.243777,0.34262,0.34262,0.243777,0.000000,0.34262,0.243777,0.68524
1,0.448321,0.00000,0.00000,0.448321,0.630099,0.00000,0.448321,0.00000


# Cosine similarity

In NLP, Cosine similarity is a metric used to measure how similar the documents are.


$$ \text{cosine similarity} = \frac{A \cdot B}{\|A\| \times \|B\|} $$

Where:  
$ A \cdot B $  = dot product of vectors A and B  
$ \|A\| $ = magnitude (length) of vector A  
$ \|B\| $ =  magnitude (length) of vector B

#### Intuition

- If vectors point in the **same direction**, cosine similarity = **1** (maximum similarity).
- If vectors are **orthogonal (90° apart)**, cosine similarity = **0** (no similarity).

<img src="https://drive.google.com/uc?export=view&id=1b-o8CVfHBsjUGY_hXmxevU91gHidIuxO" width="900">

In [7]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [8]:
# Sample text data (replace with your own documents)

doc_1 = "Data is the oil of the digital economy"
doc_2 = "Data is a new oil"

data = [doc_1, doc_2]

In [9]:
tfidf_vectorizer = TfidfVectorizer() # Create a CountVectorizer instance
vector_matrix = tfidf_vectorizer.fit_transform(data) # Fit and transform the documents into numerical vectors

In [10]:
# Calculate the cosine similarity between the documents
cosine_similarity_matrix = cosine_similarity(vector_matrix)

df_cosine = pd.DataFrame(data=cosine_similarity_matrix, index=data, columns=data)

df_cosine

,Data is the oil of the digital economy,Data is a new oil
Data is the oil of the digital economy,1.000000,0.327871
Data is a new oil,0.327871,1.000000


# 2- What is Word2vec?

Word2Vec consists of models for generating word embedding. These models are two-layer neural networks having one input layer, one hidden layer, and one output layer.

Word2Vec utilizes two architectures :
1. CBOW (Continuous Bag of Words)
2. **Skip Gram**
<img src="https://drive.google.com/uc?export=view&id=1K38nEu_KhgtSJuG2RySwc6U5AAjVAgWZ" width="600">


Run this command in terminal to install
> pip install gensim

We will use fake and real news dataset to do our expirament. You can find the dataset here: https://www.kaggle.com/datasets/clmentbisaillon/fake-and-real-news-dataset?select=True.csv


In [11]:
import pandas as pd
import nltk
import numpy as np
import gensim

from nltk.tokenize import word_tokenize

In [12]:
tokens = []

for i in df['text']:
    token = word_tokenize(i)
    tokens.append(token)

In [13]:
w2v = gensim.models.Word2Vec(tokens, min_count=1, vector_size=100, window=5, sg=1)

In [14]:
print("Cosine similarity between 'provide' and 'program' - Skip Gram : ",w2v.wv.similarity('provide', 'program'))

Cosine similarity between 'provide' and 'program' - Skip Gram :  0.42995837


In [15]:
print("words that similar to 'program' - Skip Gram : ",w2v.wv.most_similar('program'))

words that similar to 'program' - Skip Gram :  [('programme', 0.8122132420539856), ('programs', 0.8021042346954346), ('nuclear', 0.6807528734207153), ('curtail', 0.6764351725578308), ('restraints', 0.6717228889465332), ('curbed', 0.6692667603492737), ('EB-5', 0.6666446328163147), ('bond-buying', 0.6633234024047852), ('modernization', 0.6618486046791077), ('state-federal', 0.6600683927536011)]


# Tasks

### Task 1: Cosine Similarity
Use the Cosine Similarity method to determine how similar the following sentences are.

'This is the first document.',  
'This document is the second document.',  
'And this is the third one.',  
'Is this the first document?'  


In [16]:
sentences = [
    'This is the first document.',
    'This document is the second document.',
    'And this is the third one.',
    'Is this the first document?'
]

# Convert sentences to TF-IDF vectors
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(sentences)

# Calculate cosine similarity
cosine_matrix = cosine_similarity(tfidf_matrix)

# Display results
df_cosine = pd.DataFrame(
    cosine_matrix,
    index=['Sentence 1', 'Sentence 2', 'Sentence 3', 'Sentence 4'],
    columns=['Sentence 1', 'Sentence 2', 'Sentence 3', 'Sentence 4']
)

df_cosine

,Sentence 1,Sentence 2,Sentence 3,Sentence 4
Sentence 1,1.000000,0.646926,0.307772,1.000000
Sentence 2,0.646926,1.000000,0.225240,0.646926
Sentence 3,0.307772,0.225240,1.000000,0.307772
Sentence 4,1.000000,0.646926,0.307772,1.000000


In [17]:
# Make a copy and remove the diagonal
similarity = cosine_matrix.copy()
np.fill_diagonal(similarity, -1)

# Find the most similar pair
i, j = np.unravel_index(np.argmax(similarity), similarity.shape)

print("Most similar sentences:")
print("Sentence", i + 1, ":", sentences[i])
print("Sentence", j + 1, ":", sentences[j])
print("Cosine similarity:", cosine_matrix[i, j])

Most similar sentences:
Sentence 1 : This is the first document.
Sentence 4 : Is this the first document?
Cosine similarity: 1.0


### Task 2: TF-IDF
Use tf-idf method on the sentences below to determine the important words.

'data science is one of the most important fields of science',  
'this is one of the best data science courses',  
'data scientists analyze data'  


In [18]:
sentences = [
    'data science is one of the most important fields of science',
    'this is one of the best data science courses',
    'data scientists analyze data'
]

tfidf = TfidfVectorizer()

tfidf_matrix = tfidf.fit_transform(sentences)

# Create DataFrame
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=tfidf.get_feature_names_out()
)

tfidf_df

,analyze,best,courses,data,fields,important,is,most,of,one,science,scientists,the,this
0,0.000000,0.000000,0.000000,0.189526,0.320895,0.320895,0.244049,0.320895,0.488098,0.244049,0.488098,0.000000,0.244049,0.000000
1,0.000000,0.400294,0.400294,0.236420,0.000000,0.000000,0.304434,0.000000,0.304434,0.304434,0.304434,0.000000,0.304434,0.400294
2,0.542701,0.000000,0.000000,0.641055,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.542701,0.000000,0.000000


In [19]:
feature_names = tfidf.get_feature_names_out()

for i, sentence in enumerate(sentences):
    scores = tfidf_matrix[i].toarray().flatten()

    word_scores = pd.DataFrame({
        'Word': feature_names,
        'TF-IDF': scores
    })

    word_scores = word_scores[word_scores['TF-IDF'] > 0]
    word_scores = word_scores.sort_values(
        by='TF-IDF',
        ascending=False
    )

    print(f"\nSentence {i + 1}:")
    print(sentence)
    print("\nImportant words:")
    print(word_scores.to_string(index=False))


Sentence 1:
data science is one of the most important fields of science

Important words:
     Word   TF-IDF
  science 0.488098
       of 0.488098
     most 0.320895
   fields 0.320895
important 0.320895
      one 0.244049
       is 0.244049
      the 0.244049
     data 0.189526

Sentence 2:
this is one of the best data science courses

Important words:
   Word   TF-IDF
   best 0.400294
courses 0.400294
   this 0.400294
     of 0.304434
     is 0.304434
science 0.304434
    one 0.304434
    the 0.304434
   data 0.236420

Sentence 3:
data scientists analyze data

Important words:
      Word   TF-IDF
      data 0.641055
   analyze 0.542701
scientists 0.542701


In [20]:
for i, sentence in enumerate(sentences):
    scores = tfidf_matrix[i].toarray().flatten()

    max_index = np.argmax(scores)

    print(
        f"Sentence {i + 1}: "
        f"'{feature_names[max_index]}' "
        f"(TF-IDF = {scores[max_index]:.4f})"
    )

Sentence 1: 'of' (TF-IDF = 0.4881)
Sentence 2: 'best' (TF-IDF = 0.4003)
Sentence 3: 'data' (TF-IDF = 0.6411)


## Word2vec

### Task 3:

Download the Simpsons dataset **(simpsons_script_lines.csv)** and apply the preprocessing procedure.  
Use the **'spoken_words'** column.
```
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[0-9]", '', text)
    text = re.sub(r"[)(,”“.’$-]", '', text)
    return text
```
Create a skip gram Word2Vec model as below.
```
Skip_gram_model = gensim.models.Word2Vec(tokens, min_count = 1, vector_size = 100, window = 5, sg = 1)

In [21]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("prashant111/the-simpsons-dataset")

print("Path to dataset files:", path)

Path to dataset files: /root/.cache/kagglehub/datasets/prashant111/the-simpsons-dataset/versions/1


In [22]:
df = pd.read_csv(
    '/kaggle/input/the-simpsons-dataset/simpsons_script_lines.csv',
    encoding='utf-8',
    on_bad_lines='skip'
)

df.head()

/tmp/ipykernel_1724/2510428397.py:1: DtypeWarning: Columns (4,5,6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


,id,episode_id,number,raw_text,timestamp_in_ms,speaking_line,character_id,location_id,raw_character_text,raw_location_text,spoken_words,normalized_text,word_count
0,9549,32,209,"Miss Hoover: No, actually, it was a little of ...",848000,True,464.0,3.0,Miss Hoover,Springfield Elementary School,"No, actually, it was a little of both. Sometim...",no actually it was a little of both sometimes ...,31
1,9550,32,210,Lisa Simpson: (NEAR TEARS) Where's Mr. Bergstrom?,856000,True,9.0,3.0,Lisa Simpson,Springfield Elementary School,Where's Mr. Bergstrom?,wheres mr bergstrom,3
2,9551,32,211,Miss Hoover: I don't know. Although I'd sure l...,856000,True,464.0,3.0,Miss Hoover,Springfield Elementary School,I don't know. Although I'd sure like to talk t...,i dont know although id sure like to talk to h...,22
3,9552,32,212,Lisa Simpson: That life is worth living.,864000,True,9.0,3.0,Lisa Simpson,Springfield Elementary School,That life is worth living.,that life is worth living,5
4,9553,32,213,Edna Krabappel-Flanders: The polls will be ope...,864000,True,40.0,3.0,Edna Krabappel-Flanders,Springfield Elementary School,The polls will be open from now until the end ...,the polls will be open from now until the end ...,33


In [23]:
print(df.columns)

print("\nNumber of rows:", len(df))

print("\nMissing values:")
print(df['spoken_words'].isna().sum())

df['spoken_words'].head(10)

Index(['id', 'episode_id', 'number', 'raw_text', 'timestamp_in_ms',
       'speaking_line', 'character_id', 'location_id', 'raw_character_text',
       'raw_location_text', 'spoken_words', 'normalized_text', 'word_count'],
      dtype='object')

Number of rows: 158271

Missing values:
26159


,spoken_words
0,"No, actually, it was a little of both. Sometim..."
1,Where's Mr. Bergstrom?
2,I don't know. Although I'd sure like to talk t...
3,That life is worth living.
4,The polls will be open from now until the end ...
5,I don't think there's anything left to say.
6,Bart?
7,Victory party under the slide!
8,NaN
9,Mr. Bergstrom! Mr. Bergstrom!


In [24]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[0-9]", '', text)
    text = re.sub(r"[)(,”“.’$-]", '', text)
    return text

In [25]:
df['cleaned_text'] = df['spoken_words'].apply(clean_text)

df[['spoken_words', 'cleaned_text']].head(10)

,spoken_words,cleaned_text
0,"No, actually, it was a little of both. Sometim...",no actually it was a little of both sometimes ...
1,Where's Mr. Bergstrom?,where's mr bergstrom?
2,I don't know. Although I'd sure like to talk t...,i don't know although i'd sure like to talk to...
3,That life is worth living.,that life is worth living
4,The polls will be open from now until the end ...,the polls will be open from now until the end ...
5,I don't think there's anything left to say.,i don't think there's anything left to say
6,Bart?,bart?
7,Victory party under the slide!,victory party under the slide!
8,NaN,nan
9,Mr. Bergstrom! Mr. Bergstrom!,mr bergstrom! mr bergstrom!


In [26]:
tokens = []

for text in df['cleaned_text']:
    token = word_tokenize(text)
    tokens.append(token)

print("Number of sentences:", len(tokens))
print("\nFirst tokenized sentence:")
print(tokens[0])

Number of sentences: 158271

First tokenized sentence:
['no', 'actually', 'it', 'was', 'a', 'little', 'of', 'both', 'sometimes', 'when', 'a', 'disease', 'is', 'in', 'all', 'the', 'magazines', 'and', 'all', 'the', 'news', 'shows', 'it', "'s", 'only', 'natural', 'that', 'you', 'think', 'you', 'have', 'it']


In [27]:
tokens = [sentence for sentence in tokens if len(sentence) > 0]

print("Number of non-empty sentences:", len(tokens))

Number of non-empty sentences: 158255


In [28]:
Skip_gram_model = gensim.models.Word2Vec(
    tokens,
    min_count=1,
    vector_size=100,
    window=5,
    sg=1
)

print("Skip-Gram Word2Vec model created successfully.")

Skip-Gram Word2Vec model created successfully.


In [29]:
print("Vocabulary size:", len(Skip_gram_model.wv))

print("\nFirst 20 words:")
print(list(Skip_gram_model.wv.key_to_index.keys())[:20])

Vocabulary size: 44427

First 20 words:
['i', '!', 'you', 'the', '?', 'a', "'s", 'to', 'nan', 'it', 'and', 'that', 'of', "n't", 'is', 'do', 'my', 'in', 'we', 'this']


### Task 4

Use: wv.most_similar() method to :

1.	Find the words similar to “homer”.

2.	Find the words similar to “marge”.

3. Find the words similar to “bart”



In [30]:
print("Words similar to 'homer':")

similar_homer = Skip_gram_model.wv.most_similar('homer', topn=10)

for word, score in similar_homer:
    print(f"{word}: {score:.4f}")

Words similar to 'homer':
abe: 0.8741
marge: 0.8459
bart: 0.8423
eliza: 0.8209
grampa: 0.8154
sweetheart: 0.8121
bartholomew: 0.8072
laddie: 0.8065
jay: 0.8063
apu: 0.8021


In [31]:
print("Words similar to 'marge':")

similar_marge = Skip_gram_model.wv.most_similar('marge', topn=10)

for word, score in similar_marge:
    print(f"{word}: {score:.4f}")

Words similar to 'marge':
abe: 0.8522
homer: 0.8459
sweetie: 0.8273
sweetheart: 0.8243
honey: 0.8170
becky: 0.8170
allison: 0.8121
jessica: 0.8104
lurleen: 0.8087
carefully: 0.8038


In [32]:
print("Words similar to 'bart':")

similar_bart = Skip_gram_model.wv.most_similar('bart', topn=10)

for word, score in similar_bart:
    print(f"{word}: {score:.4f}")

Words similar to 'bart':
milhouse: 0.8789
lisa: 0.8542
laddie: 0.8443
abe: 0.8433
homer: 0.8423
grampa: 0.8390
jessica: 0.8385
eliza: 0.8331
stampy: 0.8290
sweetheart: 0.8268


### Task 5

Use the wv.doesnt_match() method to :

1.	Find which of 'jimbo', 'milhouse’, and 'kearney’ does not belong to the list.

3.	Find the odd one among "nelson", "bart", and "milhouse".

4.	Find the odd one among ‘homer', 'patty', and ‘selma'.

*Hint: You need to pass the strings as List*

In [33]:
words = ['jimbo', 'milhouse', 'kearney']

odd_word = Skip_gram_model.wv.doesnt_match(words)

print("Words:", words)
print("Odd word:", odd_word)

Words: ['jimbo', 'milhouse', 'kearney']
Odd word: milhouse


In [34]:
words = ['nelson', 'bart', 'milhouse']

odd_word = Skip_gram_model.wv.doesnt_match(words)

print("Words:", words)
print("Odd word:", odd_word)

Words: ['nelson', 'bart', 'milhouse']
Odd word: nelson


In [35]:
words = ['homer', 'patty', 'selma']

odd_word = Skip_gram_model.wv.doesnt_match(words)

print("Words:", words)
print("Odd word:", odd_word)

Words: ['homer', 'patty', 'selma']
Odd word: homer


In [36]:
groups = {
    "Group 1": ['jimbo', 'milhouse', 'kearney'],
    "Group 2": ['nelson', 'bart', 'milhouse'],
    "Group 3": ['homer', 'patty', 'selma']
}

for name, words in groups.items():
    odd_word = Skip_gram_model.wv.doesnt_match(words)

    print(f"{name}")
    print("Words:", words)
    print("Odd word:", odd_word)
    print("-" * 40)

Group 1
Words: ['jimbo', 'milhouse', 'kearney']
Odd word: milhouse
----------------------------------------
Group 2
Words: ['nelson', 'bart', 'milhouse']
Odd word: nelson
----------------------------------------
Group 3
Words: ['homer', 'patty', 'selma']
Odd word: homer
----------------------------------------
